<a href="https://colab.research.google.com/github/Martinhostorm/spbd-2627/blob/main/docs/labs/proj/SPBD2627_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPBD Project
## Battery Energy Savings
### 26/27

version 1.0


## Overview

This project asks you to analyse the real energy consumption of eleven residential customers in Portugal and determine how much money each customer could save by installing a home battery.

Energy costs in Portugal depend on which tariff a customer has contracted. The three available tariffs — *Simples*, *Bi-Horária*, and *Tri-Horária* — price electricity differently depending on the time of day. A well-sized battery can arbitrage these price differences by charging when electricity is cheap and discharging when it is expensive.&#32;

The analysis should be grounded in real metered data and based on Portuguese tariff rules. Your conclusions must be supported by the data, not by assumptions.


## Datasets

You will work with the following information:

**Energy meter readings** — provided as a [single Parquet file](https://raw.githubusercontent.com/smduarte/spbd-2627/main/docs/labs/proj/readings.parquet.tar) containing all 11 customers. Each row records a date, an energy counter value in kWh, and a customer identifier. The counter is monotonically increasing; the energy consumed between two readings is the difference of their counter values. Readings are irregularly spaced and some samples are missing, so for each 15-minute billing interval you must estimate the energy consumed using linear interpolation between the two nearest available readings.

**Simplified Portuguese energy tariffs:**

| Tariff | Description |
| --- | --- |
| Simples | Same price at all hours |
| Bi-Horária | Two prices: cheaper late night when the demand is lower. |
| Tri-Horária | Three prices: Similar to Bi-Horária but very expensive around times of highest demand (lunch/dinner); This tariff is mandatory for meters above a certain power rating. |

### Pricing

| Prices of active energy | Time period | €/kWh&#32; |
| --- | --- | --- |
| Simples&#32; |  | 0.1654 |
| Bi-Horária | Fora de vazio | 0.1988 |
| Bi-Horária | Vazio | 0.1087 |
| Tri-Horária | Ponta | 0.2495 |
| Tri-Horária | Cheias | 0.1690 |
| Tri-Horária | Vazio | 0.1087 |

### Time periods

|  | **Vazio** | **Fora de vazio** |
| --- | --- | --- |
| **Bi-Horária** | 22:00 às 08:00 | 08:00 às 22:00 |

|  | **Vazio** | **Cheias** | **Ponta** |
| --- | --- | --- | --- |
| **Tri-Horária** | 22:00 às 08:00 | 08:00 às 10:30 | 10:30 às 13:00 |
|  |  | 13:00 às 19:30 | 19:30 às 21:00 |
|  |  | 21:00 às 22:00 |  |

**Note:** For simplicity, we will consider that all prices will stay constant and apply uniformly across everyday of the week. We also ignore that there are different time-periods boundaries for winter and summer.


## Core Objectives

The project has two main objectives and an open-ended extension track.

### Objective 1 — Tariff Comparison & Customer Segmentation

For each of the 11 customers, use *Tri-Horária* as the baseline tariff and compute their annual electricity cost under it. Then compute the same cost under *Simples* and *Bi-Horária* and express each as a saving or surcharge relative to the *Tri-Horária* bill. This framing reflects the real constraint that *Tri-Horária* is mandatory for customers above a certain meter power rating — for those customers the alternative tariffs are not available, so the relative figures quantify what they cannot save; for the remaining customers, the comparison reveals whether switching away from *Tri-Horária* would be worthwhile.

As part of this objective, characterise the consumption patterns of the 11 customers and determine whether they fall into natural groups (e.g. daytime-heavy vs. night-heavy, peaky vs. flat, high vs. low overall consumption). Use these groups to explain why certain customers benefit more from time-of-use tariffs than others, and to set up a meaningful comparison for the battery analysis in Objective 2.

### Objective 2 — Battery Sizing

Model a battery storage system for each customer. The battery charges when the price of electricity is low (off-peak) and discharges to cover consumption when prices are high (peak or super-peak). Evaluate battery capacities in 5 kWh increments (e.g. 5, 10, 15, 20 kWh) and report the annual cost saving relative to the *Tri-Horária* baseline from Objective 1.

Use the following cost model to assess economic viability:

- **Total capital cost** = X + k × Y, where X = 1,250 € (inverter), Y = 850 € (cost per 5 kWh storage module), and k is the number of modules
- **Battery lifetime**: 10 years; ignore capacity degradation over that period
- **Payback period** = total capital cost ÷ annual saving
- **Charge/discharge power limit (kW):** 8
- **Round-trip efficiency:** 90%;
- **Useable Battery Capacity**:** 80%

For each customer, identify the capacity at which the marginal saving from adding one more module no longer reduces the payback period, and report the capacity that minimises payback time. Assume that the contracted tariff while evaluating the battery impact is *Tri-Horária.*


## Further Exploration (Optional)

After completing the two core objectives, students are encouraged to pursue one or more of the following directions. These are not exhaustive — well-motivated ideas beyond this list are welcome.

**Sensitivity analysis.** Electricity tariff prices and hardware costs change over time. How sensitive is the payback period to changes in energy prices or battery costs? Present a range of scenarios.

**Real-time spot prices.** The core analysis assumes fixed time-of-use tariff prices. In practice, Portugal's electricity market also exposes a spot price (OMIE/MIBEL), published at 15-minute resolution, that fluctuates with supply and demand. Rather than implementing a full analysis, discuss the strategies by which this information could be incorporated into the battery dispatch model — for example, threshold-based rules, day-ahead price forecasting, or optimisation approaches — and reflect on the trade-offs each strategy implies in terms of complexity, data requirements, and potential benefit. For context, explore the terms: Continuous Intraday Market (SIDC) and Grid Imbalance Market (15-Minute Settlement).

**Renter turnover detection.** Assuming the 11 customers are rental properties, produce a time-series chart of daily or weekly energy consumption for each property over the full period and discuss whether it is possible to identify moments where the occupant may have changed. A sustained drop to near-zero consumption followed by a return to normal levels is a candidate signal — however, care must be taken not to conflate a renter change with an extended absence such as a holiday, which can produce a similar pattern. Discuss what additional features of the consumption signal might help distinguish between the two cases, and reflect on the limitations of drawing such conclusions from meter data alone.

**Processing Frameworks.** For Objective 1, present solution for more than one processing framework: Spark dataframes, Spark SQl, DuckDB. Discuss the performance results obtained in line with possible production-scale implementation, targeting the full customer base of an energy provider, such as EDP Comercial.

## Under Specification

Where the problem is under-specified, students are expected to make reasonable simplifying assumptions. Each assumption must be clearly stated in the report and briefly justified.


## Deliverables

Groups submit the following.

**Code** — well-structured, runnable code (Jupyter Python notebook) covering data ingestion, the 15-minute interpolation, tariff cost calculation, and the battery simulation. A README must explain how to reproduce every result in the report. Should also include AI prompting used to develop the code or explore possible approaches.

**Analysis report** (at most 10 pages) — structured as follows:

1. Data exploration: summary statistics and visualisations of consumption patterns across customers.
2. Tariff comparison: per-customer results with explanation of which consumption features drive the best tariff choice.
3. Battery analysis: cost savings vs. battery size, with a recommendation per customer.
4. Optional extension: if pursued, a focused section presenting methodology, findings, and limitations.
5. Conclusions: key findings, limitations of the models, and suggestions for a production-scale implementation, targeting the full customer base of an energy provider, such as EDP Comercial.


## Assessment Criteria

| Criterion | Weight | What we look for |
| --- | --- | --- |
| Data engineering | 15% | Correct interpolation; handling of missing samples; reproducible pipeline |
| Tariff analysis | 25% | Accurate cost calculation; clear explanation of per-customer differences |
| Battery modelling | 30% | Sound simulation logic; appropriate range of configurations evaluated; honest treatment of assumptions |
| Visualisation & communication | 10% | Charts that make the findings legible; report and presentation accessible to a non-specialist |
| Critical thinking | 20% | Limitations acknowledged; results questioned; optional extension or original angle pursued with rigour |

Groups will be penalised for hardcoded results that cannot be reproduced by running the submitted code, and for conclusions that contradict the data without explanation.


## Group Composition

Groups can have **a maximum of 3 members**, no exceptions, where **at most 2 are human**. Groups are free to use AI tooling as a member of the team, but must declare it and report the prompting used. Group composition should be credible given the scope and quality of the work presented. Using AI is not a detrimental factor per se. Some groups may be called to Q&A session as part of the grading process.


## Deadline:

**December 9, 23:59**

Submissions after the deadline will incur a penalty of 0.25 points (on a 20-point scale) per day late. The penalty accumulates until the grade reaches 10.0, at which point no further deduction is applied. The hard deadline, beyond which no submission will be accepted, is January 4, 2027.


# Helper Code

In [ ]:
#@title Download the Readings Dataset
!curl -sS https://raw.githubusercontent.com/smduarte/spbd-2627/main/docs/labs/proj/readings.parquet.tar  --output - | tar xf -

In [ ]:
#@title PySpark for Energy readings dataset
from pyspark.sql import SparkSession

# Initialize a PySpark session
spark = SparkSession.builder \
    .appName("SPBD Project Analysis") \
    .getOrCreate()

# Path to the downloaded readings dataset directory
parquet_path = "readings.parquet"

# Read the parquet files
readings = spark.read.parquet(parquet_path)

# Print the schema
readings.printSchema()

# Show the first 5 rows
readings.show(5, truncate=False)

In [ ]:
#@title Duckdb for Energy readings dataset
import duckdb

# Query the Parquet files directly using DuckDB
# Note: DuckDB automatically reads hive-partitioned folders (like Meter=Casa_I/)
# We can include the partition column 'Meter' in our query
con = duckdb.connect()

query = """
SELECT *
FROM read_parquet('readings.parquet/**/*.parquet', filename=false)
LIMIT 5;
"""
readings = con.execute(query).df()

display( readings )

Trabalho final de sspb

nº77896

A.I. usiliar utilizada Gemini

In [ ]:
# 1. Descarregar e extrair o dataset
!curl -sS https://raw.githubusercontent.com/smduarte/spbd-2627/main/docs/labs/proj/readings.parquet.tar --output - | tar xf -

import glob
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Verificar ficheiros parquet extraídos
parquet_files = glob.glob("readings.parquet/**/*.parquet", recursive=True)
if not parquet_files:
    parquet_files = glob.glob("readings.parquet/*.parquet", recursive=True)

print(f"Ficheiros Parquet encontrados: {len(parquet_files)}")

con = duckdb.connect()
sample_df = con.execute("SELECT * FROM read_parquet('readings.parquet/**/*.parquet') LIMIT 10").df()
print("\nColunas detetadas:", list(sample_df.columns))
display(sample_df.head())

In [ ]:
def detect_schema_columns(df: pd.DataFrame):
    cols_lower = {c.lower(): c for c in df.columns}

    meter_col = next((cols_lower[k] for k in ["meter", "customer_id", "meter_id", "client_id", "id"] if k in cols_lower), None)
    time_col = next((cols_lower[k] for k in ["timestamp", "date", "ts", "datetime", "reading_time"] if k in cols_lower), None)
    val_col = next((cols_lower[k] for k in ["value", "counter", "kwh", "energy", "reading", "active_energy"] if k in cols_lower), None)

    if not all([meter_col, time_col, val_col]):
        raise ValueError(f"Não foi possível detetar todas as colunas automaticamente. Colunas disponíveis: {list(df.columns)}")

    return meter_col, time_col, val_col


def interpolate_15min_consumption(raw_df: pd.DataFrame) -> pd.DataFrame:
    meter_col, time_col, val_col = detect_schema_columns(raw_df)

    df = raw_df[[meter_col, time_col, val_col]].copy()
    df.columns = ["meter", "timestamp", "counter"]
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_localize(None)
    df["counter"] = pd.to_numeric(df["counter"], errors="coerce")
    df = df.dropna(subset=["meter", "timestamp", "counter"])

    # Ordenar por contador e timestamp; remover timestamps duplicados mantendo o máximo do contador
    df = df.sort_values(["meter", "timestamp", "counter"])
    df = df.groupby(["meter", "timestamp"], as_index=False)["counter"].max()

    # Garantir monotonicidade não-decrescente por contador
    df["counter"] = df.groupby("meter")["counter"].cummax()

    all_intervals = []

    for meter_id, grp in df.groupby("meter", sort=True):
        grp = grp.set_index("timestamp").sort_index()

        # Alinhar início e fim para múltiplos exatos de 15 minutos
        start_ts = grp.index.min().ceil("15min")
        end_ts = grp.index.max().floor("15min")
        if end_ts <= start_ts:
            continue

        grid_15m = pd.date_range(start=start_ts, end=end_ts, freq="15min")

        # Unir leituras reais e grelha de 15 min, interpolar linearmente no tempo e filtrar para a grelha
        combined_index = grp.index.union(grid_15m).sort_values()
        combined = grp.reindex(combined_index)
        combined["counter_interp"] = combined["counter"].interpolate(method="time", limit_area="inside")

        grid_df = combined.loc[grid_15m, ["counter_interp"]].dropna().copy()

        # Energia consumida no intervalo de 15 min anterior até grid_15m (kWh)
        grid_df["consumption_kwh"] = grid_df["counter_interp"].diff().clip(lower=0.0)
        grid_df = grid_df.iloc[1:].copy()  # primeira linha não tem intervalo anterior completo
        grid_df["meter"] = meter_id
        grid_df["interval_end"] = grid_df.index
        grid_df["interval_start"] = grid_df["interval_end"] - pd.Timedelta(minutes=15)

        all_intervals.append(grid_df[["meter", "interval_start", "interval_end", "consumption_kwh"]].reset_index(drop=True))

    panel_15m = pd.concat(all_intervals, ignore_index=True)
    return panel_15m


raw_readings = con.execute("SELECT * FROM read_parquet('readings.parquet/**/*.parquet')").df()
df_15m = interpolate_15min_consumption(raw_readings)
print(f"Total de intervalos de 15 min gerados: {len(df_15m):,} | Clientes: {df_15m['meter'].nunique()}")
display(df_15m.head())

In [ ]:
# 1. Preparar o período mensal a partir do dataframe já tarifado (df_priced)
df_monthly_both = df_priced.copy()
df_monthly_both["year_month"] = df_monthly_both["interval_start"].dt.to_period("M").astype(str)

# 2. Agregar consumo (kWh) e custo (€) por casa e mês
# (Aqui usamos cost_tri como referência; podes trocar por cost_simples ou cost_bi se quiseres)
agg_monthly = (
    df_monthly_both.groupby(["meter", "year_month"], as_index=False)[["consumption_kwh", "cost_tri"]]
    .sum()
)

# 3. Criar as duas tabelas pivot separadas
piv_kwh = agg_monthly.pivot(index="meter", columns="year_month", values="consumption_kwh").fillna(0.0)
piv_cost = agg_monthly.pivot(index="meter", columns="year_month", values="cost_tri").fillna(0.0)

# Ordenar colunas cronologicamente e adicionar coluna de Total
months_sorted = sorted(piv_kwh.columns)
piv_kwh = piv_kwh[months_sorted]
piv_cost = piv_cost[months_sorted]

piv_kwh["Total"] = piv_kwh.sum(axis=1)
piv_cost["Total"] = piv_cost.sum(axis=1)

# 4. Intercalar as linhas: para cada casa, 1ª linha = Consumo (kWh), 2ª linha = Custo (€)
rows_data = []
row_index = []

for meter_id in sorted(piv_kwh.index):
    rows_data.append(piv_kwh.loc[meter_id].round(2))
    row_index.append((meter_id, "Consumo (kWh)"))

    rows_data.append(piv_cost.loc[meter_id].round(2))
    row_index.append((meter_id, "Custo Tri-Horária (€)"))

multi_idx = pd.MultiIndex.from_tuples(row_index, names=["Casa", "Métrica"])
df_monthly_interleaved = pd.DataFrame(rows_data, index=multi_idx)

print("=== CONSUMO (kWh) E CUSTO TRI-HORÁRIA (€) MENSAL POR CASA ===")
display(df_monthly_interleaved)

In [ ]:
PRICES = {
    "simples": 0.1654,
    "bi_vazio": 0.1087,
    "bi_fora_vazio": 0.1988,
    "tri_vazio": 0.1087,
    "tri_cheias": 0.1690,
    "tri_ponta": 0.2495,
}


def classify_tariff_periods(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    minutes_from_midnight = out["interval_start"].dt.hour * 60 + out["interval_start"].dt.minute

    # Vazio: 22:00 (1320 min) às 08:00 (480 min)
    is_vazio = (minutes_from_midnight >= 22 * 60) | (minutes_from_midnight < 8 * 60)

    # Ponta (Tri-Horária): 10:30 (630) às 13:00 (780) e 19:30 (1170) às 21:00 (1260)
    is_ponta_tri = (
        ((minutes_from_midnight >= 10 * 60 + 30) & (minutes_from_midnight < 13 * 60)) |
        ((minutes_from_midnight >= 19 * 60 + 30) & (minutes_from_midnight < 21 * 60))
    )

    out["period_bi"] = np.where(is_vazio, "Vazio", "Fora de vazio")
    out["period_tri"] = np.where(
        is_vazio,
        "Vazio",
        np.where(is_ponta_tri, "Ponta", "Cheias")
    )

    out["price_simples"] = PRICES["simples"]
    out["price_bi"] = np.where(is_vazio, PRICES["bi_vazio"], PRICES["bi_fora_vazio"])
    out["price_tri"] = np.where(
        is_vazio,
        PRICES["tri_vazio"],
        np.where(is_ponta_tri, PRICES["tri_ponta"], PRICES["tri_cheias"])
    )

    out["cost_simples"] = out["consumption_kwh"] * out["price_simples"]
    out["cost_bi"] = out["consumption_kwh"] * out["price_bi"]
    out["cost_tri"] = out["consumption_kwh"] * out["price_tri"]
    return out


df_priced = classify_tariff_periods(df_15m)

# Resumo por cliente e anualização (escala para 365 dias com base na amplitude temporal observada)
tariff_rows = []
for meter_id, grp in df_priced.groupby("meter", sort=True):
    span_days = max((grp["interval_end"].max() - grp["interval_start"].min()).total_seconds() / 86400.0, 1.0)
    annual_factor = 365.0 / span_days

    total_kwh = grp["consumption_kwh"].sum()
    annual_kwh = total_kwh * annual_factor

    c_tri_ann = grp["cost_tri"].sum() * annual_factor
    c_sim_ann = grp["cost_simples"].sum() * annual_factor
    c_bi_ann = grp["cost_bi"].sum() * annual_factor

    # Percentagens de consumo por período da Tri-Horária
    pct_vazio = grp.loc[grp["period_tri"] == "Vazio", "consumption_kwh"].sum() / max(total_kwh, 1e-9) * 100
    pct_cheias = grp.loc[grp["period_tri"] == "Cheias", "consumption_kwh"].sum() / max(total_kwh, 1e-9) * 100
    pct_ponta = grp.loc[grp["period_tri"] == "Ponta", "consumption_kwh"].sum() / max(total_kwh, 1e-9) * 100

    # Rácio pico/média (peakiness) nos intervalos de 15 min
    mean_15m = grp["consumption_kwh"].mean()
    peak_ratio = grp["consumption_kwh"].max() / max(mean_15m, 1e-9)

    tariff_rows.append({
        "meter": meter_id,
        "observed_days": round(span_days, 1),
        "annual_kwh": round(annual_kwh, 1),
        "cost_tri_yr_eur": round(c_tri_ann, 2),
        "cost_simples_yr_eur": round(c_sim_ann, 2),
        "saving_simples_vs_tri_eur": round(c_tri_ann - c_sim_ann, 2),
        "cost_bi_yr_eur": round(c_bi_ann, 2),
        "saving_bi_vs_tri_eur": round(c_tri_ann - c_bi_ann, 2),
        "share_vazio_pct": round(pct_vazio, 1),
        "share_cheias_pct": round(pct_cheias, 1),
        "share_ponta_pct": round(pct_ponta, 1),
        "peak_to_mean_15m": round(peak_ratio, 2),
    })

df_tariffs = pd.DataFrame(tariff_rows)

# Segmentação simples e interpretável baseada no volume anual e perfil horário
med_kwh = df_tariffs["annual_kwh"].median()
def segment_customer(row):
    vol_tag = "Alto Consumo" if row["annual_kwh"] >= med_kwh else "Baixo Consumo"
    if row["share_vazio_pct"] >= 45.0:
        time_tag = "Noturno / Vazio-Forte"
    elif row["share_ponta_pct"] >= 16.0:
        time_tag = "Ponta-Intensivo"
    else:
        time_tag = "Diurno / Equilibrado"
    return f"{vol_tag} | {time_tag}"

df_tariffs["segment"] = df_tariffs.apply(segment_customer, axis=1)

print("=== OBJETIVO 1: COMPARAÇÃO TARIFÁRIA ANUALIZADA E SEGMENTAÇÃO ===")
display(df_tariffs)

In [ ]:
BATTERY_CAPACITIES_KWH = [5, 10, 15, 20, 25, 30]
INVERTER_COST_EUR = 1250.0
MODULE_COST_EUR = 850.0
MODULE_SIZE_KWH = 5.0
MAX_POWER_KW = 8.0
INTERVAL_HOURS = 0.25  # 15 minutos
MAX_STEP_KWH = MAX_POWER_KW * INTERVAL_HOURS  # 2.0 kWh
ROUND_TRIP_EFF = 0.90
EFF_ONE_WAY = np.sqrt(ROUND_TRIP_EFF)
USABLE_FRACTION = 0.80
LIFETIME_YEARS = 10


def simulate_battery_series(load_kwh: np.ndarray, prices_tri: np.ndarray, periods_tri: np.ndarray, capacity_kwh: float):
    """
    Simula a operação da bateria em passos de 15 minutos sob a tarifa Tri-Horária.
    - Carrega em períodos de 'Vazio' (preço mais baixo) até à capacidade útil.
    - Descarrega em períodos de 'Ponta' e 'Cheias' para abater o consumo quando o preço
      efetivo compensa a eficiência round-trip (p_discharge * ROUND_TRIP_EFF > p_vazio).
    """
    usable_cap = capacity_kwh * USABLE_FRACTION
    p_vazio = PRICES["tri_vazio"]

    soc = 0.0  # energia armazenada útil na bateria (kWh)
    n = len(load_kwh)
    battery_cost_series = np.zeros(n, dtype=np.float64)

    for i in range(n):
        load = load_kwh[i]
        price = prices_tri[i]
        period = periods_tri[i]

        if period == "Vazio":
            # Carregar a partir da rede até ao limite de potência e capacidade útil
            space_in_bat = max(0.0, usable_cap - soc)
            max_stored_this_step = min(MAX_STEP_KWH * EFF_ONE_WAY, space_in_bat)
            grid_charge_kwh = max_stored_this_step / EFF_ONE_WAY

            soc += max_stored_this_step
            # Custo na rede: consumo próprio + energia para carregar a bateria
            battery_cost_series[i] = (load + grid_charge_kwh) * price

        elif period in ("Ponta", "Cheias") and (price * ROUND_TRIP_EFF > p_vazio):
            # Descarregar para abater o consumo local quando compensa economicamente
            max_bat_release = min(MAX_STEP_KWH, soc)
            max_grid_offset_from_bat = max_bat_release * EFF_ONE_WAY

            delivered_to_load = min(load, max_grid_offset_from_bat)
            bat_drawn = delivered_to_load / EFF_ONE_WAY

            soc = max(0.0, soc - bat_drawn)
            remaining_grid_load = max(0.0, load - delivered_to_load)
            battery_cost_series[i] = remaining_grid_load * price

        else:
            # Sem carga/descarga: apenas consumo direto da rede
            battery_cost_series[i] = load * price

    return battery_cost_series


battery_results = []

for meter_id, grp in df_priced.groupby("meter", sort=True):
    span_days = max((grp["interval_end"].max() - grp["interval_start"].min()).total_seconds() / 86400.0, 1.0)
    annual_factor = 365.0 / span_days

    load_arr = grp["consumption_kwh"].to_numpy(dtype=np.float64)
    price_arr = grp["price_tri"].to_numpy(dtype=np.float64)
    period_arr = grp["period_tri"].to_numpy()

    base_tri_cost_yr = grp["cost_tri"].sum() * annual_factor

    prev_payback = np.inf
    marginal_stop_cap = None

    for cap in BATTERY_CAPACITIES_KWH:
        k_modules = cap / MODULE_SIZE_KWH
        capex = INVERTER_COST_EUR + k_modules * MODULE_COST_EUR

        bat_cost_series = simulate_battery_series(load_arr, price_arr, period_arr, cap)
        bat_tri_cost_yr = bat_cost_series.sum() * annual_factor
        annual_saving = base_tri_cost_yr - bat_tri_cost_yr

        payback = (capex / annual_saving) if annual_saving > 1e-6 else np.inf
        saving_10yr_net = annual_saving * LIFETIME_YEARS - capex

        # Identificar capacidade a partir da qual adicionar mais um módulo deixa de reduzir o payback
        if marginal_stop_cap is None and payback >= prev_payback:
            marginal_stop_cap = cap
        prev_payback = payback

        battery_results.append({
            "meter": meter_id,
            "capacity_kwh": cap,
            "modules_k": int(k_modules),
            "capex_eur": round(capex, 2),
            "annual_saving_eur": round(annual_saving, 2),
            "payback_years": round(payback, 2) if np.isfinite(payback) else np.inf,
            "net_saving_10yr_eur": round(saving_10yr_net, 2),
        })

df_battery = pd.DataFrame(battery_results)

# Resumo por cliente: capacidade que minimiza o payback e ponto onde deixa de reduzir
summary_rows = []
for meter_id, sub in df_battery.groupby("meter", sort=True):
    finite_sub = sub[np.isfinite(sub["payback_years"])]
    if not finite_sub.empty:
        best_row = finite_sub.loc[finite_sub["payback_years"].idxmin()]
        best_cap = int(best_row["capacity_kwh"])
        min_pb = float(best_row["payback_years"])
        best_saving = float(best_row["annual_saving_eur"])
    else:
        best_cap, min_pb, best_saving = None, np.inf, 0.0

    # Capacidade onde adicionar módulo deixa de melhorar o payback
    sub_sorted = sub.sort_values("capacity_kwh").reset_index(drop=True)
    no_improve_cap = None
    for idx in range(1, len(sub_sorted)):
        if sub_sorted.loc[idx, "payback_years"] >= sub_sorted.loc[idx - 1, "payback_years"]:
            no_improve_cap = int(sub_sorted.loc[idx, "capacity_kwh"])
            break

    summary_rows.append({
        "meter": meter_id,
        "min_payback_capacity_kwh": best_cap,
        "min_payback_years": round(min_pb, 2) if np.isfinite(min_pb) else np.inf,
        "annual_saving_at_best_eur": round(best_saving, 2),
        "no_payback_reduction_at_kwh": no_improve_cap if no_improve_cap is not None else f">{max(BATTERY_CAPACITIES_KWH)}",
    })

df_battery_summary = pd.DataFrame(summary_rows)
print("=== OBJETIVO 2: RESUMO DE RECOMENDAÇÃO DE BATERIA POR CLIENTE ===")
display(df_battery_summary)

In [ ]:
meters_sorted = sorted(df_tariffs["meter"].unique())
cmap = plt.get_cmap("tab20")
meter_colors = {m: cmap(i % 20) for i, m in enumerate(meters_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))

# 1. Gráfico de Poupança / Sobretaxa anual face à Tri-Horária
df_tariffs_plot = df_tariffs.set_index("meter").loc[meters_sorted].reset_index()
x = np.arange(len(df_tariffs_plot))
width = 0.38
axes[0].bar(
    x - width / 2,
    df_tariffs_plot["saving_simples_vs_tri_eur"],
    width,
    label="Simples vs Tri-Horária",
)
axes[0].bar(
    x + width / 2,
    df_tariffs_plot["saving_bi_vs_tri_eur"],
    width,
    label="Bi-Horária vs Tri-Horária",
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_tariffs_plot["meter"], rotation=45, ha="right")
axes[0].set_ylabel("Poupança / Sobretaxa Anual (€/ano)")
axes[0].set_title("Objetivo 1: Impacto Tarifário vs. Tri-Horária")
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# 2. Gráfico Mensal de Consumo (kWh) — mesma cor por casa
monthly_plot_df = (
    df_15m.assign(year_month=df_15m["interval_start"].dt.to_period("M").astype(str))
    .groupby(["meter", "year_month"], as_index=False)["consumption_kwh"]
    .sum()
    .pivot(index="year_month", columns="meter", values="consumption_kwh")
    .fillna(0.0)
)
monthly_plot_df = monthly_plot_df.loc[sorted(monthly_plot_df.index)]

for m in meters_sorted:
    if m in monthly_plot_df.columns:
        axes[1].plot(
            monthly_plot_df.index,
            monthly_plot_df[m],
            marker="o",
            color=meter_colors[m],
            label=str(m),
        )

axes[1].set_xlabel("Mês (YYYY-MM)")
axes[1].set_ylabel("Consumo Mensal (kWh)")
axes[1].set_title("Consumo Mensal por Casa")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(True, linestyle="--", alpha=0.4)

# 3. Gráfico de Payback vs. Capacidade da Bateria — mesma cor por casa
for m in meters_sorted:
    sub = df_battery[df_battery["meter"] == m].sort_values("capacity_kwh")
    axes[2].plot(
        sub["capacity_kwh"],
        sub["payback_years"],
        marker="o",
        color=meter_colors[m],
        label=str(m),
    )

axes[2].axhline(LIFETIME_YEARS, color="red", linestyle="--", label="Vida útil (10 anos)")
axes[2].set_xlabel("Capacidade da Bateria (kWh)")
axes[2].set_ylabel("Payback Period (anos)")
axes[2].set_title("Objetivo 2: Payback vs. Capacidade da Bateria")
axes[2].grid(True, linestyle="--", alpha=0.4)

# Legenda única partilhada para as casas
handles, labels = axes[2].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.05),
    ncol=min(len(labels), 7),
    fontsize="small",
)

plt.tight_layout()
plt.show()